# 04 · Evaluation et analyse des erreurs

Le meilleur modèle est choisi sur la perte de validation. Le test n'est utilisé qu'après ce choix.

In [ ]:
from pathlib import Path
import sys
candidates = [Path.cwd(), Path.cwd().parent, Path('/content/SceneSense')]
ROOT = next((p for p in candidates if (p / 'pyproject.toml').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Open this notebook from the SceneSense project folder in Colab or Jupyter')
sys.path.insert(0, str(ROOT / 'src'))
from scenesense.config import Settings, CLASSES
settings = Settings(data_dir=ROOT / "data/intel-image-classification", output_dir=ROOT / "results")
print("Project:", ROOT)
print("Dataset:", settings.data_dir)

## Résumé de la validation

In [ ]:
from scenesense.evaluation import load_validation_results, plot_training, select_model, evaluate_final
import pandas as pd
from IPython.display import display
summary = load_validation_results(settings).sort_values("validation_loss")
display(summary[["experiment", "validation_accuracy", "validation_loss", "trainable_parameters", "duration_seconds"]])
print("Model chosen by validation loss:", select_model(settings))

In [ ]:
from PIL import Image
path = plot_training(settings)
display(Image.open(path))

## Test final

N'exécuter cette étape qu'une fois les comparaisons de validation terminées.

In [ ]:
final = evaluate_final(settings)
print(final)

## Matrice de confusion et erreurs

In [ ]:
from pathlib import Path
best = final["experiment"]
folder = settings.output_dir / "evaluation" / f"test_{best}"
display(Image.open(folder / "confusion_matrix.png"))
print((folder / "classification_report.txt").read_text())
errors = pd.read_csv(folder / "predictions.csv")
errors = errors[~errors.correct].sort_values("confidence", ascending=False)
display(errors.head(10))

In [ ]:
import matplotlib.pyplot as plt
if not errors.empty:
    fig, axes = plt.subplots(2, 3, figsize=(11, 7))
    for ax, (_, row) in zip(axes.flat, errors.head(6).iterrows()):
        with Image.open(settings.data_dir / row.path) as im:
            ax.imshow(im.convert("RGB"))
        ax.set_title(f"True: {CLASSES[int(row.label)]} / Pred: {CLASSES[int(row.prediction)]}", fontsize=9)
        ax.axis("off")
    for ax in axes.flat[len(errors.head(6)):]:
        ax.axis("off")
    plt.tight_layout()
    plt.show()
else:
    print("No classification errors found in the test split.")